# Build model-ready features and data splits
This notebook converts a raw peptide dataset into a training-ready package.

Main tasks:
1. Build feature representations (for example ECFP fingerprints, graph features, etc.).
2. Generate multiple train/validation/test split strategies.

In [1]:
def summarize_feature_building(save_dir, dataset_name):
    """Summarize generated artifacts for one dataset.

    Args:
        save_dir (str): Output directory of generated files.
        dataset_name (str): Dataset name for logging.

    Returns:
        dict: Mapping from file name to file metadata.
    """
    import glob
    import os

    print("\n" + "=" * 60)
    print("Feature building summary")
    print("=" * 60)

    generated_files = glob.glob(os.path.join(save_dir, "*"))
    print(f"Output directory: {save_dir}")
    print("Generated files:")

    file_info = {}
    for file_path in sorted(generated_files):
        filename = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        file_size_kb = file_size / 1024
        print(f"  - {filename} ({file_size_kb:.1f} KB)")
        file_info[filename] = {
            "size_bytes": file_size,
            "size_kb": file_size_kb,
            "path": file_path,
        }

    print(f"\nDataset '{dataset_name}' feature building completed.")
    print("You can now use these artifacts for downstream experiments.")

    return file_info

In [2]:
from numpy import mean
from pepbenchmark.splitter.hybrid_splitter import HybridSplitter


def build_data_splits(sequences, labels, save_dir, n_splits=5):
    """Build and persist multiple data split strategies with logging.

    Args:
        sequences (list[str]): Peptide sequence list.
        labels (list): Label list aligned with `sequences`.
        save_dir (str): Output directory.
        n_splits (int): Number of random seeds/split repeats.

    Returns:
        dict: {'splits': {...}, 'errors': [...]}
    """
    import datetime
    import json
    import os
    from pepbenchmark.splitter.cdhit_splitter import CDHitSplitter
    from pepbenchmark.splitter.mmseq_splitter import MMseqs2Splitter
    from pepbenchmark.splitter.motif_splitter import MotifSplitter
    from pepbenchmark.splitter.random_splitter import RandomSplitter

    os.makedirs(save_dir, exist_ok=True)
    log_file = os.path.join(save_dir, "data_splits_log.txt")

    def log_and_print(message):
        print(message)
        with open(log_file, "a", encoding="utf-8") as f:
            ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            f.write(f"{ts} - {message}\n")

    def save_json(file_name, payload):
        with open(os.path.join(save_dir, file_name), "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False)

    with open(log_file, "w", encoding="utf-8") as f:
        f.write("Data split build log\n")
        f.write(f"Start time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 80 + "\n\n")

    results = {"splits": {}, "errors": []}
    log_and_print(f"Building splits for {len(sequences)} sequences...")
    log_and_print(f"Parameters: n_splits={n_splits}")

    try:
        random_splitter = RandomSplitter()
        cdhit_splitter = CDHitSplitter()
        mmseqs_splitter = MMseqs2Splitter()
        motif_splitter = MotifSplitter()
        hybrid_splitter = HybridSplitter()
        log_and_print("Splitter initialization succeeded")
    except Exception as e:
        error_msg = f"Splitter initialization failed: {e}"
        log_and_print(error_msg)
        results["errors"].append(error_msg)
        return results

    split_jobs = [
        (
            "random_split",
            random_splitter,
            "random_split.json",
            {},
            "Random",
        ),
        (
            "cdhit_split",
            cdhit_splitter,
            "cdhit_split.json",
            {"c": 0.4},
            "CD-HIT",
        ),
        (
            "mmseqs2_split",
            mmseqs_splitter,
            "mmseqs2_split.json",
            {
                "min_seq_id": 0.3,
                "c": 0.6,
                "cov_mode": 0,
                "alignment_mode": 3,
                "seq_id_mode": 2,
                "s": 8,
                "kmer_per_seq": 50,
            },
            "MMseqs2",
        ),
    ]

    for split_key, splitter, out_file, extra_kwargs, split_name in split_jobs:
        try:
            log_and_print(f"Generating {split_name} split...")
            split_res = splitter.get_split_indices_n(
                n_splits=n_splits,
                data=sequences,
                labels=labels,
                frac_train=0.8,
                frac_valid=0.1,
                frac_test=0.1,
                **extra_kwargs,
            )
            save_json(out_file, split_res)
            results["splits"][split_key] = split_res
            log_and_print(f"  {split_name} split done ({n_splits} seeds)")
        except Exception as e:
            error_msg = f"{split_name} split failed: {e}"
            log_and_print(f"  {error_msg}")
            results["errors"].append(error_msg)

    avg_len = mean([len(seq) for seq in sequences]) if sequences else 0
    ks_hybrid = 2 if avg_len < 15 else 5
    ks_motif = 3 if avg_len < 15 else 5

    try:
        log_and_print("Generating Hybrid split...")
        hybrid_res = hybrid_splitter.get_split_indices_n(
            n_splits=n_splits,
            data=sequences,
            labels=labels,
            frac_train=0.8,
            frac_valid=0.1,
            frac_test=0.1,
            cluster_distribution_strategy="",
            preserve_cluster_integrity=True,
            seed=42,
            ks=ks_hybrid,
            test_method="fisher",
            alternative="greater",
            min_cluster_size=3,
            min_pos=3,
            mode="all",
            pval_threshold=0.05,
            min_score=4.0,
            fdr_correct=True,
            topM=None,
            top_fraction=None,
            min_seq_id=0.3,
            c=0.6,
            cov_mode=0,
            alignment_mode=3,
            seq_id_mode=2,
            s=8,
            kmer_per_seq=50,
        )
        save_json("hybrid_split.json", hybrid_res)
        results["splits"]["hybrid_split"] = hybrid_res
        log_and_print(f"  Hybrid split done ({n_splits} seeds)")
    except Exception as e:
        error_msg = f"Hybrid split failed: {e}"
        log_and_print(f"  {error_msg}")
        results["errors"].append(error_msg)

    try:
        log_and_print("Generating motif split...")
        motif_res = motif_splitter.get_split_indices_n(
            n_splits=n_splits,
            data=sequences,
            labels=labels,
            frac_train=0.8,
            frac_valid=0.1,
            frac_test=0.1,
            cluster_distribution_strategy="size_aware",
            preserve_cluster_integrity=True,
            seed=42,
            ks=ks_motif,
            test_method="fisher",
            alternative="greater",
            min_cluster_size=3,
            topM=10000,
            top_fraction=None,
            min_support=3,
            min_jaccard=0.6,
        )
        save_json("motif_split.json", motif_res)
        results["splits"]["motif_split"] = motif_res
        log_and_print(f"  Motif split done ({n_splits} seeds)")
    except Exception as e:
        error_msg = f"Motif split failed: {e}"
        log_and_print(f"  {error_msg}")
        results["errors"].append(error_msg)

    if results["errors"]:
        log_and_print("\nEncountered errors:")
        for error in results["errors"]:
            log_and_print(f"  - {error}")

    log_and_print("\nData split generation completed")
    log_and_print(f"Log file: {log_file}")
    return results


def build_all_features(
    sequences,
    labels,
    save_dir,
    dataset_name,
    include_esm2=True,
    include_splits=False,
    batch_size=32,
    n_splits=5,
):
    """Build all supported features for a peptide dataset.

    Args:
        sequences (list[str]): Sequence list.
        labels (list): Label list aligned with `sequences`.
        save_dir (str): Output directory.
        dataset_name (str): Dataset name for logging.
        include_esm2 (bool): Whether to build ESM2 embeddings.
        include_splits (bool): Whether to also build data splits.
        batch_size (int): Reserved for embedding batching settings.
        n_splits (int): Number of repeats for split generation.

    Returns:
        dict: Build outputs, generated splits, and errors.
    """
    import os

    import numpy as np
    import pandas as pd
    import torch
    from pepbenchmark.pep_utils.convert import (
        Fasta2Biln,
        Fasta2Embedding,
        Fasta2Helm,
        Fasta2Smiles,
        Smiles2FP,
        Smiles2Graph,
    )

    print(f"Start building features for dataset '{dataset_name}'")
    print(f"Dataset size: {len(sequences)}")

    os.makedirs(save_dir, exist_ok=True)

    results = {
        "features": {},
        "errors": [],
        "file_info": {},
        "splits": {},
    }

    print("1. Saving base sequence/label files...")
    try:
        pd.DataFrame({"feature": sequences}).to_csv(
            os.path.join(save_dir, "fasta.csv"), index=False
        )
        pd.DataFrame({"feature": labels}).to_csv(
            os.path.join(save_dir, "label.csv"), index=False
        )
        results["features"]["fasta"] = sequences
        results["features"]["label"] = labels
        print("   Base files saved")
    except Exception as e:
        error_msg = f"Failed to save base files: {e}"
        print(f"   {error_msg}")
        results["errors"].append(error_msg)

    print("2. Initializing converters...")
    try:
        fasta2smiles = Fasta2Smiles()
        fasta2helm = Fasta2Helm()
        fasta2biln = Fasta2Biln()
        fasta2embedding = (
            Fasta2Embedding(model="facebook/esm2_t30_150M_UR50D")
            if include_esm2
            else None
        )
        print("   Converters initialized")
    except Exception as e:
        error_msg = f"Converter initialization failed: {e}"
        print(f"   {error_msg}")
        results["errors"].append(error_msg)
        return results

    print("3. Building SMILES features...")
    smiles_list = None
    try:
        smiles_list = fasta2smiles(sequences)
        pd.DataFrame({"feature": smiles_list}).to_csv(
            os.path.join(save_dir, "smiles.csv"), index=False
        )
        results["features"]["smiles"] = smiles_list
        print(f"   Generated {len(smiles_list)} SMILES entries")
    except Exception as e:
        error_msg = f"SMILES generation failed: {e}"
        print(f"   {error_msg}")
        results["errors"].append(error_msg)

    print("4. Building HELM features...")
    try:
        helm_list = fasta2helm(sequences)
        pd.DataFrame({"feature": helm_list}).to_csv(
            os.path.join(save_dir, "helm.csv"), index=False
        )
        results["features"]["helm"] = helm_list
        print(f"   Generated {len(helm_list)} HELM entries")
    except Exception as e:
        error_msg = f"HELM generation failed: {e}"
        print(f"   {error_msg}")
        results["errors"].append(error_msg)

    print("5. Building BILN features...")
    try:
        biln_list = fasta2biln(sequences)
        pd.DataFrame({"feature": biln_list}).to_csv(
            os.path.join(save_dir, "biln.csv"), index=False
        )
        results["features"]["biln"] = biln_list
        print(f"   Generated {len(biln_list)} BILN entries")
    except Exception as e:
        error_msg = f"BILN generation failed: {e}"
        print(f"   {error_msg}")
        results["errors"].append(error_msg)

    if smiles_list is not None:
        print("6. Building molecular fingerprint features...")
        try:
            ecfp4_features = Smiles2FP(fp_type="Morgan", radius=2, nBits=1024)(smiles_list)
            ecfp4_array = np.array(ecfp4_features)
            np.savez_compressed(os.path.join(save_dir, "ecfp4.npz"), data=ecfp4_array)
            results["features"]["ecfp4"] = ecfp4_array
            print(f"   ECFP4 shape: {ecfp4_array.shape}")
        except Exception as e:
            error_msg = f"ECFP4 generation failed: {e}"
            print(f"   {error_msg}")
            results["errors"].append(error_msg)

        try:
            ecfp6_features = Smiles2FP(fp_type="Morgan", radius=3, nBits=1024)(smiles_list)
            ecfp6_array = np.array(ecfp6_features)
            np.savez_compressed(os.path.join(save_dir, "ecfp6.npz"), data=ecfp6_array)
            results["features"]["ecfp6"] = ecfp6_array
            print(f"   ECFP6 shape: {ecfp6_array.shape}")
        except Exception as e:
            error_msg = f"ECFP6 generation failed: {e}"
            print(f"   {error_msg}")
            results["errors"].append(error_msg)
    else:
        print("6. Skip fingerprint features (SMILES unavailable)")

    if smiles_list is not None:
        print("7. Building graph features...")
        try:
            graph_features = Smiles2Graph()(smiles_list)
            torch.save(graph_features, os.path.join(save_dir, "graph.pt"))
            results["features"]["graph"] = graph_features
            print(f"   Graph objects: {len(graph_features)}")
        except Exception as e:
            error_msg = f"Graph feature generation failed: {e}"
            print(f"   {error_msg}")
            results["errors"].append(error_msg)
    else:
        print("7. Skip graph features (SMILES unavailable)")

    if include_esm2:
        print("8. Building ESM2 embedding features...")
        print(f"   This may take time. Requested batch_size={batch_size}")
        try:
            embeddings = fasta2embedding(sequences)
            embeddings_tensor = (
                torch.tensor(embeddings)
                if not isinstance(embeddings, torch.Tensor)
                else embeddings
            )
            torch.save(
                embeddings_tensor,
                os.path.join(save_dir, "esm2_150_embedding.pt"),
            )
            results["features"]["esm2_150_embedding"] = embeddings_tensor
            print(f"   ESM2 embedding shape: {embeddings_tensor.shape}")
        except Exception as e:
            error_msg = f"ESM2 embedding generation failed: {e}"
            print(f"   {error_msg}")
            results["errors"].append(error_msg)
    else:
        print("8. Skip ESM2 embedding features")

    if include_splits:
        print("9. Building data splits...")
        try:
            split_results = build_data_splits(
                sequences=sequences,
                labels=labels,
                save_dir=save_dir,
                n_splits=n_splits,
            )
            results["splits"] = split_results["splits"]
            if split_results["errors"]:
                results["errors"].extend(split_results["errors"])
            print(f"   Built {len(split_results['splits'])} split types")
        except Exception as e:
            error_msg = f"Split generation failed: {e}"
            print(f"   {error_msg}")
            results["errors"].append(error_msg)
    else:
        print("9. Skip data split generation")

    results["file_info"] = summarize_feature_building(save_dir, dataset_name)
    return results

/home/batchcom/assist/miniforge3/envs/pepbenchmark/lib/python3.10/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: cdhit
2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: mmseqs2
2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: motif
2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: kmer
2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: similarity
2026-03-28 15:42:39 | INFO     | pepbenchmark.cluster.factory | Registered clustering method: molecular


In [3]:
import os
import pandas as pd

dataset_name = "dppiv_inhibitors"
data = pd.read_csv(f"./{dataset_name}/all.csv")

data["length"] = data["sequence"].apply(len)
raw_max_length = data["length"].max()
print(f"[{dataset_name}] original max length: {raw_max_length}")

# Optional filter: keep peptides with length <= 50
data = data[data["length"] <= 50].copy()
filtered_max_length = data["length"].max()
print(f"[{dataset_name}] max length after filtering: {filtered_max_length}")

save_dir = f"./{dataset_name}/"
os.makedirs(save_dir, exist_ok=True)
data[["sequence"]].to_csv(os.path.join(save_dir, "fasta.csv"), index=False)

sequences = data["sequence"].tolist()
labels = data["label"].tolist()
print(f"Total samples after filtering: {len(sequences)}")

results = build_all_features(
    sequences=sequences,
    labels=labels,
    save_dir=save_dir,
    dataset_name=dataset_name,
    include_esm2=False,
    include_splits=True,
    batch_size=128,
    n_splits=5,
 )

[dppiv_inhibitors] original max length: 33
[dppiv_inhibitors] max length after filtering: 33
Total samples after filtering: 1268
Start building features for dataset 'dppiv_inhibitors'
Dataset size: 1268
1. Saving base sequence/label files...
   Base files saved
2. Initializing converters...
   Converters initialized
3. Building SMILES features...


Converting FASTA to SMILES: 100%|██████████| 1268/1268 [00:00<00:00, 3184.13it/s]


   Generated 1268 SMILES entries
4. Building HELM features...


Converting FASTA to HELM: 100%|██████████| 1268/1268 [00:00<00:00, 32691.26it/s]


DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides
DEBUG: No connections to process - this is normal for linear peptides
DEBUG: No explicit bonds found in parsed data - this is normal for linear peptides

...

   Generated 1268 BILN entries
6. Building molecular fingerprint features...


Generating Morgan fingerprints: 100%|██████████| 1268/1268 [00:00<00:00, 3374.47it/s]


   ECFP4 shape: (1268, 1024)


Generating Morgan fingerprints: 100%|██████████| 1268/1268 [00:00<00:00, 3217.25it/s]


   ECFP6 shape: (1268, 1024)
7. Building graph features...
   Graph objects: 1268
8. Skip ESM2 embedding features
9. Building data splits...
Building splits for 1268 sequences...
Parameters: n_splits=5
2026-03-28 15:42:47 | INFO     | RandomSplitter | RandomSplitter initialized
2026-03-28 15:42:47 | INFO     | CDHitSplitter | CDHitSplitter initialized
2026-03-28 15:42:47 | INFO     | MMseqs2Splitter | MMseqs2Splitter initialized
2026-03-28 15:42:47 | INFO     | MotifSplitter | MotifSplitter initialized
2026-03-28 15:42:47 | INFO     | pepbenchmark.splitter.hybrid_splitter | HybridSplitter initialized
Splitter initialization succeeded
Generating Random split...
2026-03-28 15:42:47 | INFO     | pepbenchmark.splitter.base_splitter | Generating 5 splits
2026-03-28 15:42:47 | INFO     | pepbenchmark.splitter.base_splitter | Generating split 1/5 with seed 42
2026-03-28 15:42:47 | INFO     | pepbenchmark.splitter.random_splitter | Starting random split: data_size=1268, frac_train=0.8, frac_va